# Nashville Housing Data Cleaning

Initial data loading, inspection, and setup for the Nashville Housing dataset.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

## 1. Load the data

In [2]:
# read in the raw data and create a copy for cleaning
RAW_PATH = Path("../data/raw/Nashville Housing Data for Data Cleaning.xlsx")
df_raw = pd.read_excel(RAW_PATH)
df_clean = df_raw.copy()

# create a dataframe to log changes
change_log = pd.DataFrame(columns=['step', 'description', 'rows_affected'])

# add a function to log changes to the change_log dataframe
def log_change(step: str, description: str, rows_affected: int) -> None:
    """Append one row to change_log (step, description, rows_affected)."""
    change_log.loc[len(change_log)] = [step, description, int(rows_affected)]

In [3]:
print("--- df.head() ---")
display(df_clean.head())

--- df.head() ---


,UniqueID,ParcelID,LandUse,PropertyAddress,SaleDate,SalePrice,LegalReference,SoldAsVacant,OwnerName,OwnerAddress,Acreage,TaxDistrict,LandValue,BuildingValue,TotalValue,YearBuilt,Bedrooms,FullBath,HalfBath
0,2045,007 00 0 125.00,SINGLE FAMILY,"1808 FOX CHASE DR, GOODLETTSVILLE",2013-04-09,240000,20130412-0036474,No,"FRAZIER, CYRENTHA LYNETTE","1808 FOX CHASE DR, GOODLETTSVILLE, TN",2.3,GENERAL SERVICES DISTRICT,50000.0,168200.0,235700.0,1986.0,3.0,3.0,0.0
1,16918,007 00 0 130.00,SINGLE FAMILY,"1832 FOX CHASE DR, GOODLETTSVILLE",2014-06-10,366000,20140619-0053768,No,"BONER, CHARLES & LESLIE","1832 FOX CHASE DR, GOODLETTSVILLE, TN",3.5,GENERAL SERVICES DISTRICT,50000.0,264100.0,319000.0,1998.0,3.0,3.0,2.0
2,54582,007 00 0 138.00,SINGLE FAMILY,"1864 FOX CHASE DR, GOODLETTSVILLE",2016-09-26,435000,20160927-0101718,No,"WILSON, JAMES E. & JOANNE","1864 FOX CHASE DR, GOODLETTSVILLE, TN",2.9,GENERAL SERVICES DISTRICT,50000.0,216200.0,298000.0,1987.0,4.0,3.0,0.0
3,43070,007 00 0 143.00,SINGLE FAMILY,"1853 FOX CHASE DR, GOODLETTSVILLE",2016-01-29,255000,20160129-0008913,No,"BAKER, JAY K. & SUSAN E.","1853 FOX CHASE DR, GOODLETTSVILLE, TN",2.6,GENERAL SERVICES DISTRICT,50000.0,147300.0,197300.0,1985.0,3.0,3.0,0.0
4,22714,007 00 0 149.00,SINGLE FAMILY,"1829 FOX CHASE DR, GOODLETTSVILLE",2014-10-10,278000,20141015-0095255,No,"POST, CHRISTOPHER M. & SAMANTHA C.","1829 FOX CHASE DR, GOODLETTSVILLE, TN",2.0,GENERAL SERVICES DISTRICT,50000.0,152300.0,202300.0,1984.0,4.0,3.0,0.0


In [4]:
print("--- df_clean.info() ---")
df_clean.info()

--- df_clean.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 56477 entries, 0 to 56476
Data columns (total 19 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   UniqueID         56477 non-null  int64         
 1   ParcelID         56477 non-null  str           
 2   LandUse          56477 non-null  str           
 3   PropertyAddress  56448 non-null  str           
 4   SaleDate         56477 non-null  datetime64[us]
 5   SalePrice        56477 non-null  int64         
 6   LegalReference   56477 non-null  object        
 7   SoldAsVacant     56477 non-null  str           
 8   OwnerName        25261 non-null  str           
 9   OwnerAddress     26015 non-null  str           
 10  Acreage          26015 non-null  float64       
 11  TaxDistrict      26015 non-null  str           
 12  LandValue        26015 non-null  float64       
 13  BuildingValue    26015 non-null  float64       
 14  TotalValue       26015 no

In [5]:
print("--- df.shape ---")
print(df_clean.shape)

--- df.shape ---
(56477, 19)


In [6]:
assert df_clean.shape == (56477, 19), f"Unexpected shape: {df_clean.shape}"
assert 'df_raw' in globals() and 'df_clean' in globals(), "Dataframes not initialized"
print("Initial loading and inspection assertions passed successfully.")

Initial loading and inspection assertions passed successfully.


## 2. Data Profiling

In [7]:
# Missing values: count and percentage per column
missing = pd.DataFrame({
    "missing_count": df_clean.isna().sum(),
    "missing_pct": (df_raw.isna().mean() * 100).round(2),
})
missing[missing["missing_count"] > 0].sort_values("missing_count", ascending=False)

,missing_count,missing_pct
HalfBath,32333,57.25
Bedrooms,32320,57.23
YearBuilt,32314,57.22
FullBath,32202,57.02
OwnerName,31216,55.27
TotalValue,30462,53.94
OwnerAddress,30462,53.94
Acreage,30462,53.94
LandValue,30462,53.94
BuildingValue,30462,53.94


In [8]:
# Duplicates at three levels of strictness
print("Fully identical rows:", df_raw.duplicated().sum())
print("Repeated UniqueID:", df_raw["UniqueID "].duplicated().sum())

# Same sale recorded more than once (business-key duplicates)
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
print("Duplicates on business key:", df_raw.duplicated(subset=key_cols).sum())

Fully identical rows: 0
Repeated UniqueID: 0
Duplicates on business key: 103


In [9]:
# Categorical consistency
print(df_raw["SoldAsVacant"].value_counts(dropna=False))
print(df_raw["LandUse"].nunique(), "distinct LandUse values")

SoldAsVacant
No     51403
Yes     4623
N        399
Y         52
Name: count, dtype: int64
39 distinct LandUse values


In [10]:
# Hidden whitespace in address columns
for col in ["PropertyAddress", "OwnerAddress"]:
    s = df_raw[col].dropna().astype(str)
    print(col,
          "| double spaces:", s.str.contains(r"\s{2,}").sum(),
          "| leading/trailing:", (s != s.str.strip()).sum())

PropertyAddress | double spaces: 55863 | leading/trailing: 7
OwnerAddress | double spaces: 25315 | leading/trailing: 0


In [11]:
# Numeric sanity check
df_raw.describe().T

,count,mean,min,25%,50%,75%,max,std
UniqueID,56477.0,28334.001133,0.0,14186.0,28313.0,42513.0,56635.0,16352.590651
SaleDate,56477,2015-02-08 08:02:27.373267,2013-01-02 00:00:00,2014-04-11 00:00:00,2015-03-24 00:00:00,2015-12-30 00:00:00,2019-12-13 00:00:00,NaN
SalePrice,56477.0,327226.369566,50.0,135000.0,205100.0,328500.0,54278060.0,929832.611076
Acreage,26015.0,0.498923,0.01,0.18,0.27,0.45,160.06,1.570454
LandValue,26015.0,69068.557601,100.0,21000.0,28800.0,60000.0,2772000.0,106040.12844
BuildingValue,26015.0,160784.677109,0.0,75900.0,111400.0,180700.0,12971800.0,206799.854557
TotalValue,26015.0,232375.398847,100.0,102800.0,148500.0,268350.0,13940400.0,281064.346448
YearBuilt,24163.0,1963.744899,1799.0,1948.0,1960.0,1983.0,2017.0,26.542982
Bedrooms,24157.0,3.089912,0.0,3.0,3.0,3.0,11.0,0.852869
FullBath,24275.0,1.886014,0.0,1.0,2.0,2.0,10.0,0.961515


In [12]:
# Check 1: are the six "30,462 missing" columns missing in the SAME rows?
struct_cols = ["OwnerAddress", "Acreage", "LandValue",
               "BuildingValue", "TotalValue", "TaxDistrict"]
pattern = df_raw[struct_cols].isna()
print("Rows where ALL six are missing:", pattern.all(axis=1).sum())
print("Rows where SOME (not all) are missing:",
      (pattern.any(axis=1) & ~pattern.all(axis=1)).sum())

# Check 2: do the 103 duplicates differ ONLY by UniqueID?
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
dups = df_raw[df_raw.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
other_cols = [c for c in df_raw.columns if c not in ["UniqueID "]]
print("Duplicate rows identical except UniqueID:",
      df_raw.duplicated(subset=other_cols, keep=False).sum(), "of", len(dups))

# Check 3: how many of the 29 missing addresses can be recovered via ParcelID?
known = (df_raw.dropna(subset=["PropertyAddress"])
           .drop_duplicates("ParcelID")
           .set_index("ParcelID")["PropertyAddress"])
missing_addr = df_raw[df_raw["PropertyAddress"].isna()]
print("Recoverable via ParcelID:",
      missing_addr["ParcelID"].isin(known.index).sum(), "of", len(missing_addr))

Rows where ALL six are missing: 30462
Rows where SOME (not all) are missing: 0
Duplicate rows identical except UniqueID: 204 of 206
Recoverable via ParcelID: 29 of 29


In [13]:
# Find duplicate groups (by business key) that are NOT identical in other columns
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
dups = df_raw[df_raw.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)

# Rows identical except UniqueID are "safe" duplicates; the rest need a manual look
other_cols = [c for c in df_raw.columns if c != "UniqueID "]
safe_mask = dups.duplicated(subset=other_cols, keep=False)
odd_rows = dups[~safe_mask]

# Show the odd rows transposed so differences are easy to read
odd_rows.T

,30034,30035
UniqueID,12629,11912
ParcelID,110 03 0A 005.00,110 03 0A 005.00
LandUse,SINGLE FAMILY,VACANT RES LAND
PropertyAddress,"2616 LAKESIDE MEADOWS DR, MOUNT JULIET","2616 LAKESIDE MEADOWS DR, MOUNT JULIET"
SaleDate,2014-02-05 00:00:00,2014-02-05 00:00:00
SalePrice,65237,65237
LegalReference,20140206-0010606,20140206-0010606
SoldAsVacant,Yes,Yes
OwnerName,NaN,NaN
OwnerAddress,NaN,NaN


## Step 0 - Setup

Verify prerequisites, ensure working copy isolation, check initial shape, and validate SaleDate datetime type.

In [14]:
# Ensure dataframes exist and working copy is isolated
assert 'df_raw' in globals() and 'df_clean' in globals(), "df_raw or df_clean missing"
assert df_clean is not df_raw, "df_clean must be a copy, not the same object as df_raw"

# Ensure change_log is initialized and empty
assert 'change_log' in globals(), "change_log missing"
assert len(change_log) == 0, "change_log should be empty at Step 0"

def count_changed(before: pd.Series, after: pd.Series) -> int:
    """Number of values that actually changed; NaN -> NaN counts as unchanged."""
    both_nan = before.isna() & after.isna()
    return int(((before != after) & ~both_nan).sum())

# Verify shape
print(f"Initial shape: {df_clean.shape}")
assert df_clean.shape == (56477, 19), f"Unexpected shape: {df_clean.shape}"

# Verify SaleDate dtype
is_dt = pd.api.types.is_datetime64_any_dtype(df_clean['SaleDate'])
print(f"SaleDate is datetime: {is_dt}")
assert is_dt, "SaleDate is not datetime type"

print("Step 0 Setup assertions passed successfully.")

Initial shape: (56477, 19)
SaleDate is datetime: True
Step 0 Setup assertions passed successfully.


## Step 1 - Text Standardization

Standardize text columns by stripping leading/trailing whitespace, collapsing repeated spaces, fixing LandUse categories and typos, and renaming 'UniqueID ' to 'UniqueID'.

In [15]:
# Step 1a: Standardize PropertyAddress and OwnerAddress whitespace
def clean_address(series):
    return series.str.strip().str.replace(r'\s+', ' ', regex=True)

prop_before = df_clean['PropertyAddress'].copy()
owner_before = df_clean['OwnerAddress'].copy()

df_clean['PropertyAddress'] = clean_address(df_clean['PropertyAddress'])
df_clean['OwnerAddress'] = clean_address(df_clean['OwnerAddress'])

prop_changed = count_changed(prop_before, df_clean['PropertyAddress'])
owner_changed = count_changed(owner_before, df_clean['OwnerAddress'])

print(f"PropertyAddress rows changed: {prop_changed}")
print(f"OwnerAddress rows changed: {owner_changed}")

log_change('Step 1a', 'Strip and collapse whitespace in PropertyAddress', prop_changed)
log_change('Step 1a', 'Strip and collapse whitespace in OwnerAddress', owner_changed)

# Assertions
assert prop_changed >= 55863, f"Unexpected PropertyAddress change count: {prop_changed}"
assert df_clean['PropertyAddress'].isna().sum() == 29, "PropertyAddress null count changed"
assert df_clean['OwnerAddress'].isna().sum() == 30462, "OwnerAddress null count changed"
print("Step 1a completed successfully.")

PropertyAddress rows changed: 55863
OwnerAddress rows changed: 25315
Step 1a completed successfully.


In [16]:
# Step 1b: Standardize LandUse categories
land_before = df_clean['LandUse'].copy()
df_clean['LandUse'] = df_clean['LandUse'].str.strip().str.replace(r'\s+', ' ', regex=True)

log_change('Step 1b', 'LandUse: strip and collapse whitespace',
           count_changed(land_before, df_clean['LandUse']))

mapping = {
    'VACANT RES LAND': 'VACANT RESIDENTIAL LAND',
    'VACANT RESIENTIAL LAND': 'VACANT RESIDENTIAL LAND',
    'RESTURANT/CAFETERIA': 'RESTAURANT/CAFETERIA'
}

mask_greenbelt = df_clean['LandUse'].str.contains('GRRENBELT', na=False)
df_clean.loc[mask_greenbelt, 'LandUse'] = 'GREENBELT/RES'

for k, v in mapping.items():
    df_clean.loc[df_clean['LandUse'] == k, 'LandUse'] = v

c1 = (land_before == 'VACANT RES LAND').sum()
c2 = (land_before == 'VACANT RESIENTIAL LAND').sum()
c3 = (land_before == 'RESTURANT/CAFETERIA').sum()
c4 = mask_greenbelt.sum()

log_change('Step 1b', "LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENTIAL LAND'", c1)
log_change('Step 1b', "LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT RESIDENTIAL LAND'", c2)
log_change('Step 1b', "LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/CAFETERIA'", c3)
log_change('Step 1b', "LandUse: 'GRRENBELT...' -> 'GREENBELT/RES'", c4)

distinct_landuse = df_clean['LandUse'].nunique()
vacant_res_count = (df_clean['LandUse'] == 'VACANT RESIDENTIAL LAND').sum()

print(f"Distinct LandUse count: {distinct_landuse}")
print(f"VACANT RESIDENTIAL LAND count: {vacant_res_count}")

assert distinct_landuse == 37, f"Unexpected distinct LandUse count: {distinct_landuse}"
assert vacant_res_count == 5099, f"Unexpected VACANT RESIDENTIAL LAND count: {vacant_res_count}"
print("Step 1b completed successfully.")

Distinct LandUse count: 37
VACANT RESIDENTIAL LAND count: 5099
Step 1b completed successfully.


In [17]:
# Step 1c: Rename 'UniqueID ' to 'UniqueID'
if 'UniqueID ' in df_clean.columns:
    df_clean.rename(columns={'UniqueID ': 'UniqueID'}, inplace=True)
    log_change('Step 1c', "Rename column header 'UniqueID ' (trailing space) -> 'UniqueID' (1 column)", 1)
    
assert 'UniqueID' in df_clean.columns and 'UniqueID ' not in df_clean.columns, "UniqueID rename failed"

# Final validation for Step 1
for col in ['PropertyAddress', 'OwnerAddress', 'LandUse']:
    s = df_clean[col].dropna().astype(str)
    assert s.str.contains(r'\s{2,}').sum() == 0, f"{col} still has double spaces"
    assert (s != s.str.strip()).sum() == 0, f"{col} still has leading/trailing spaces"
    assert not s.str.contains('_x000D_').any(), f"{col} still has carriage return artifact"

print("Step 1 text standardization completed successfully.")
display(change_log)

Step 1 text standardization completed successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1


## Step 2 - SoldAsVacant Standardization

Standardize 'SoldAsVacant' column by mapping 'Y' to 'Yes' and 'N' to 'No' for consistency.

In [18]:
# Step 2: Standardize SoldAsVacant values ('Y' -> 'Yes', 'N' -> 'No')
vacant_before = df_clean['SoldAsVacant'].copy()

mapping_vacant = {'Y': 'Yes', 'N': 'No'}
df_clean['SoldAsVacant'] = df_clean['SoldAsVacant'].replace(mapping_vacant)

vacant_changed = count_changed(vacant_before, df_clean['SoldAsVacant'])
print(f"SoldAsVacant rows changed: {vacant_changed}")

log_change('Step 2', "SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'", vacant_changed)

# Value counts after standardization
vc = df_clean['SoldAsVacant'].value_counts()
print(vc)

# Assertions
assert vacant_changed == 451, f"Unexpected changed rows count: {vacant_changed}"
assert set(df_clean['SoldAsVacant'].unique()) == {'Yes', 'No'}, "Unexpected values in SoldAsVacant"
assert vc['Yes'] == 4675, f"Unexpected Yes count: {vc['Yes']}"
assert vc['No'] == 51802, f"Unexpected No count: {vc['No']}"
assert df_clean['SoldAsVacant'].isna().sum() == 0, "Nulls introduced in SoldAsVacant"

print(
    "Step 2 SoldAsVacant standardization completed successfully.")
display(change_log)

SoldAsVacant rows changed: 451
SoldAsVacant
No     51802
Yes     4675
Name: count, dtype: int64
Step 2 SoldAsVacant standardization completed successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451


## Step 3 - Fill PropertyAddress via ParcelID

Pre-check that each ParcelID has exactly one distinct non-null PropertyAddress, then fill the 29 missing PropertyAddress values using corresponding ParcelID records.

In [19]:
# Step 3: Fill missing PropertyAddress via ParcelID
# Scoped pre-check: only the parcels of the 29 null rows matter.
null_mask = df_clean["PropertyAddress"].isna()
null_parcels = df_clean.loc[null_mask, "ParcelID"].unique()

known = df_clean.dropna(subset=["PropertyAddress"])
known_for_null = known[known["ParcelID"].isin(null_parcels)]
addr_per_parcel = known_for_null.groupby("ParcelID")["PropertyAddress"].nunique()

print("Null rows:", null_mask.sum(), "| distinct parcels:", len(null_parcels))
print("Parcels with a known address:", len(addr_per_parcel))
print("Parcels with more than one distinct address:", (addr_per_parcel > 1).sum())

assert null_mask.sum() == 29, "Expected 29 null PropertyAddress rows"
assert len(addr_per_parcel) == len(null_parcels), "STOP: a null-row parcel has no known address"
assert (addr_per_parcel == 1).all(), "STOP: a null-row parcel has several known addresses"

# Mapping is unambiguous (exactly one address per parcel, verified above)
parcel_to_addr = (known_for_null.drop_duplicates("ParcelID")
                  .set_index("ParcelID")["PropertyAddress"])

before = df_clean["PropertyAddress"].copy()
df_clean.loc[null_mask, "PropertyAddress"] = (
    df_clean.loc[null_mask, "ParcelID"].map(parcel_to_addr)
)

changed = count_changed(before, df_clean["PropertyAddress"])
log_change("Step 3", "Fill missing PropertyAddress via ParcelID (29 rows)", changed)

# Validation
assert changed == 29, f"Expected 29 rows changed, got {changed}"
assert df_clean["PropertyAddress"].isna().sum() == 0, "Nulls remain in PropertyAddress"
assert before[~null_mask].equals(df_clean.loc[~null_mask, "PropertyAddress"]), \
    "A previously known address was modified"
print("Step 3 completed successfully.")

Null rows: 29 | distinct parcels: 29
Parcels with a known address: 29
Parcels with more than one distinct address: 0
Step 3 completed successfully.


In [20]:
# Evidence for the report: parcels that appear with more than one distinct address
known = df_clean.dropna(subset=["PropertyAddress"])
addr_counts = known.groupby("ParcelID")["PropertyAddress"].nunique()
conflict_ids = addr_counts[addr_counts > 1].index

conflicts = (known[known["ParcelID"].isin(conflict_ids)]
             [["ParcelID", "PropertyAddress", "UniqueID", "SaleDate", "LegalReference"]]
             .sort_values(["ParcelID", "PropertyAddress"]))
conflicts.to_csv("../reports/parcel_address_conflicts.csv", index=False)

print(len(conflict_ids), "conflicting parcels |", len(conflicts), "rows saved")
assert len(conflict_ids) == 243

243 conflicting parcels | 526 rows saved


## Step 4 - Split Addresses

Split PropertyAddress into sparate components (Address, City) and OwnerAddress into (Address, City, State) while keeping originals.

In [21]:
# Step 4a: Split PropertyAddress into PropertySplitAddress and PropertySplitCity
prop_not_null = df_clean['PropertyAddress'].notna()
prop_parts = df_clean.loc[prop_not_null, 'PropertyAddress'].str.split(',', n=1, expand=True)

if prop_parts.shape[1] != 2:
    print("STOP: PropertyAddress did not split into exactly 2 parts!")
    display(prop_parts[prop_parts.shape[1] != 2])
    raise ValueError("Unexpected number of parts in PropertyAddress split")

df_clean['PropertySplitAddress'] = pd.Series(dtype='object')
df_clean['PropertySplitCity'] = pd.Series(dtype='object')
df_clean.loc[prop_not_null, 'PropertySplitAddress'] = prop_parts[0].str.strip()
df_clean.loc[prop_not_null, 'PropertySplitCity'] = prop_parts[1].str.strip()

prop_split_count = prop_not_null.sum()
log_change('Step 4', 'Split PropertyAddress into PropertySplitAddress and PropertySplitCity', prop_split_count)
print(f"PropertyAddress split completed for {prop_split_count} rows.")

PropertyAddress split completed for 56477 rows.


In [22]:
# Step 4b: Split OwnerAddress into OwnerSplitAddress, OwnerSplitCity, and OwnerSplitState
owner_not_null = df_clean['OwnerAddress'].notna()
owner_parts = df_clean.loc[owner_not_null, 'OwnerAddress'].str.split(',', n=2, expand=True)

if owner_parts.shape[1] != 3:
    print("STOP: OwnerAddress did not split into exactly 3 parts!")
    display(owner_parts[owner_parts.shape[1] != 3])
    raise ValueError(
        "Unexpected number of parts in OwnerAddress split")

df_clean['OwnerSplitAddress'] = pd.Series(dtype='object')
df_clean['OwnerSplitCity'] = pd.Series(dtype='object')
df_clean['OwnerSplitState'] = pd.Series(dtype='object')
df_clean.loc[owner_not_null, 'OwnerSplitAddress'] = owner_parts[0].str.strip()
df_clean.loc[owner_not_null, 'OwnerSplitCity'] = owner_parts[1].str.strip()
df_clean.loc[owner_not_null, 'OwnerSplitState'] = owner_parts[2].str.strip()

owner_split_count = owner_not_null.sum()
log_change('Step 4', 'Split OwnerAddress into OwnerSplitAddress, City, State', owner_split_count)
print(f"OwnerAddress split completed for {owner_split_count} rows.")

OwnerAddress split completed for 26015 rows.


In [23]:
# Step 4c: Validations for Step 4
assert df_clean['PropertySplitAddress'].isna().sum() == 0, "PropertySplitAddress has nulls"
assert df_clean['PropertySplitCity'].isna().sum() == 0, "PropertySplitCity has nulls"
assert df_clean['OwnerSplitAddress'].isna().sum() == 30462, "OwnerSplitAddress null count mismatch"
assert df_clean['OwnerSplitCity'].isna().sum() == 30462, "OwnerSplitCity null count mismatch"
assert df_clean['OwnerSplitState'].isna().sum() == 30462, "OwnerSplitState null count mismatch"

print("Step 4 address splitting completed and validated successfully.")
display(change_log)

Step 4 address splitting completed and validated successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


In [24]:
# Step 4d: the n=1 / n=2 splits cannot reveal extra commas, so count them directly
prop_commas = df_clean['PropertyAddress'].str.count(',')
owner_commas = df_clean['OwnerAddress'].str.count(',')
print(prop_commas.value_counts(dropna=False).to_string())
print(owner_commas.value_counts(dropna=False).to_string())
print(df_clean['OwnerSplitState'].value_counts(dropna=False).to_string())

assert (prop_commas == 1).all(), "Some PropertyAddress values do not have exactly 1 comma"
assert owner_commas.dropna().eq(2).all(), "Some OwnerAddress values do not have exactly 2 commas"

PropertyAddress
1    56477
OwnerAddress
NaN    30462
2.0    26015
OwnerSplitState
NaN    30462
TN     26015


## Step 5 - Remove Business-Key Duplicates

Identify and remove business-key duplicates, keeping the record with the lowest UniqueID in each group.

In [25]:
# Step 5a: Pre-check duplicates on business key
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
dups_mask = df_clean.duplicated(subset=key_cols, keep=False)
total_dups = dups_mask.sum()
print(f"Total rows flagged as duplicates (keep=False): {total_dups}")

# Expected: 104 extra groups of 2 (i.e. 208 rows total involved)
assert total_dups == 208, f"STOP: Expected 208 duplicate rows involved, got {total_dups}"
print("Duplicate pre-check passed successfully.")

Total rows flagged as duplicates (keep=False): 208
Duplicate pre-check passed successfully.


In [26]:
# Step 5b: Drop duplicates keeping the lowest UniqueID per group
rows_before = len(df_clean)

# Sort by UniqueID ascending so keep='first' retains the lowest UniqueID
df_clean = df_clean.sort_values("UniqueID").drop_duplicates(subset=key_cols, keep="first").reset_index(drop=True)

rows_after = len(df_clean)
rows_removed = rows_before - rows_after

print(f"Rows before: {rows_before}")
print(f"Rows after: {rows_after}")
print(f"Rows removed: {rows_removed}")

log_change("Step 5", "Remove business-key duplicates (kept lowest UniqueID)", rows_removed)

Rows before: 56477
Rows after: 56373
Rows removed: 104


In [27]:
# Step 5c: Validations for duplicate removal
assert rows_removed == 104, f"Expected 104 rows removed, got {rows_removed}"
assert rows_after == 56373, f"Expected final shape rows 56373, got {rows_after}"
assert df_clean.duplicated(subset=key_cols).sum() == 0, "Duplicates still remain on business key"
assert 11912 in df_clean['UniqueID'].values, "Expected UniqueID 11912 to be present"
assert 12629 not in df_clean['UniqueID'].values, "Expected UniqueID 12629 to be absent"
assert 26162 in df_clean['UniqueID'].values and 27140 not in df_clean['UniqueID'].values

print("Step 5 duplicate removal completed and validated successfully.")
display(change_log)

Step 5 duplicate removal completed and validated successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


## Step 6 - Flag Suspicious Values

Add boolean flag columns for suspicious/outlier values (LowSalePrice, ZeroBedrooms, LargeAcreage, UnusualYearBuilt) without deleting any rows.

In [28]:
# Step 6a: Flag_LowSalePrice (SalePrice < 10000)
df_clean['Flag_LowSalePrice'] = (df_clean['SalePrice'] < 10000).fillna(False)
c_low = df_clean['Flag_LowSalePrice'].sum()
print(f"Flag_LowSalePrice count: {c_low}")
log_change('Step 6', 'Flag low sale price (< 10000)', c_low)
assert c_low == 94, f"Expected 94 rows for Flag_LowSalePrice, got {c_low}"

Flag_LowSalePrice count: 94


In [29]:
# Step 6b: Flag_ZeroBedrooms
residential_land_uses = [
    'SINGLE FAMILY', 'RESIDENTIAL CONDO', 'CONDO', 'DUPLEX',
    'TRIPLEX', 'QUADPLEX', 'ZERO LOT LINE', 'MOBILE HOME'
]
df_clean['Flag_ZeroBedrooms'] = (
    (df_clean['Bedrooms'] == 0) &
    df_clean['LandUse'].isin(residential_land_uses)
).fillna(False)

c_zero_bed = df_clean['Flag_ZeroBedrooms'].sum()
print(f"Flag_ZeroBedrooms count: {c_zero_bed}")
print("LandUse breakdown of Flag_ZeroBedrooms == True:")
display(df_clean[df_clean['Flag_ZeroBedrooms']]['LandUse'].value_counts())

log_change('Step 6', 'Flag zero bedrooms in residential properties', c_zero_bed)
assert 4 <= c_zero_bed <= 12, f"Expected 4 to 12 rows for Flag_ZeroBedrooms, got {c_zero_bed}"

Flag_ZeroBedrooms count: 4
LandUse breakdown of Flag_ZeroBedrooms == True:


LandUse
SINGLE FAMILY    4
Name: count, dtype: int64

In [30]:
# Step 6c: Flag_LargeAcreage (Acreage > 10)
df_clean['Flag_LargeAcreage'] = (df_clean['Acreage'] > 10).fillna(False)
c_acreage = df_clean['Flag_LargeAcreage'].sum()
print(f"Flag_LargeAcreage count: {c_acreage}")
log_change('Step 6', 'Flag large acreage (> 10)', c_acreage)
assert c_acreage == 48, f"Expected 48 rows for Flag_LargeAcreage, got {c_acreage}"

Flag_LargeAcreage count: 48


In [31]:
# Step 6d: Flag_UnusualYearBuilt (YearBuilt > SaleDate.dt.year + 1)
df_clean['Flag_UnusualYearBuilt'] = (
    df_clean['YearBuilt'] > df_clean['SaleDate'].dt.year + 1
).fillna(False)

c_year = df_clean['Flag_UnusualYearBuilt'].sum()
print(f"Flag_UnusualYearBuilt count: {c_year}")
print("SoldAsVacant breakdown of Flag_UnusualYearBuilt == True:")
display(df_clean[df_clean['Flag_UnusualYearBuilt']]['SoldAsVacant'].value_counts(dropna=False))

log_change('Step 6', 'Flag unusual year built (after sale date + 1)', c_year)
assert c_year == 175, f"Expected 175 rows for Flag_UnusualYearBuilt, got {c_year}"

Flag_UnusualYearBuilt count: 175
SoldAsVacant breakdown of Flag_UnusualYearBuilt == True:


SoldAsVacant
No     125
Yes     50
Name: count, dtype: int64

In [32]:
# Step 6e: Validations for Step 6
flag_cols = ['Flag_LowSalePrice', 'Flag_ZeroBedrooms', 'Flag_LargeAcreage', 'Flag_UnusualYearBuilt']
for col in flag_cols:
    assert df_clean[col].dtype == bool, f"{col} is not boolean"
    assert df_clean[col].isna().sum() == 0, f"{col} contains nulls"

assert len(df_clean) == 56373, f"Unexpected row count: {len(df_clean)}"
print("Step 6 suspicious value flagging completed and validated successfully.")
display(change_log)

Step 6 suspicious value flagging completed and validated successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


### A note on the build-year flag

While checking `YearBuilt`, I flagged 175 homes whose recorded build year is more than one year *after* the year they were sold. Homes are normally sold after they're built, so these rows deserve a second look.

Most of them look like a normal construction timeline: for 174 of the 175, the house was "built" 2 to 3 years after the sale, not 20 or 30. And 50 of them (29%) were sold as vacant land. Among all homes that have a build year, only 1.8% were sold as vacant, so that is about 16 times higher than usual. The simplest story: an empty lot was sold, and a house was built on it later. Another 29 rows are classified as vacant residential land even though they weren't marked as sold vacant.

That leaves 87 single-family homes that don't fit this story. They could be sales made during construction, or recorded differently. The data alone can't tell us.

For that reason I only **flagged** these rows (`Flag_UnusualYearBuilt = True`). Nothing was changed or removed. Also worth knowing: `False` means "not flagged", not "checked and correct", because rows with no build year also get `False`.

## Step 7 - HasAssessorData Flag

Add boolean column 'HasAssessorData' indicating whether all six structural assessor columns are present, without imputation.

In [33]:
# Step 7a: Pre-check structural missingness pattern
struct_cols = ["OwnerAddress", "Acreage", "LandValue", "BuildingValue", "TotalValue", "TaxDistrict"]

pattern = df_clean[struct_cols].isna()
all_missing_count = pattern.all(axis=1).sum()
some_missing_count = (pattern.any(axis=1) & ~pattern.all(axis=1)).sum()

print(f"Rows where ALL six structural columns are missing: {all_missing_count}")
print(f"Rows where SOME (not all) are missing: {some_missing_count}")

assert some_missing_count == 0, "STOP: Partial missingness detected in structural columns!"
print("Structural missingness all-or-none pre-check passed successfully.")

Rows where ALL six structural columns are missing: 30404
Rows where SOME (not all) are missing: 0
Structural missingness all-or-none pre-check passed successfully.


In [34]:
# Step 7b: Create HasAssessorData boolean flag
before_null_counts = {col: df_clean[col].isna().sum() for col in struct_cols}

# True where all six structural columns are non-null
df_clean['HasAssessorData'] = df_clean[struct_cols].notna().all(axis=1)

false_count = (~df_clean['HasAssessorData']).sum()
true_count = df_clean['HasAssessorData'].sum()

print(f"HasAssessorData True count: {true_count}")
print(f"HasAssessorData False count: {false_count}")

log_change('Step 7', 'Add HasAssessorData boolean flag', false_count)

HasAssessorData True count: 25969
HasAssessorData False count: 30404


In [35]:
# Validations
assert df_clean['HasAssessorData'].dtype == bool, "HasAssessorData is not boolean"
assert len(df_clean) == 56373, f"Unexpected row count: {len(df_clean)}"
assert false_count == df_clean['TotalValue'].isna().sum(), "False count mismatch with TotalValue nulls"
assert 30358 <= false_count <= 30462, f"False count {false_count} out of expected range"

for col in struct_cols:
    assert df_clean[col].isna().sum() == before_null_counts[col], f"{col} null count changed"

assert true_count + false_count == len(df_clean), "Sum of True and False does not match total rows"

print("Step 7 HasAssessorData flag created and validated successfully.")
display(change_log)

Step 7 HasAssessorData flag created and validated successfully.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


## Step 8 - Final Validation

Perform comprehensive final validation checks on the cleaned dataframe and change log.

In [36]:
# Step 8: Comprehensive Final Validation
print(f"Final shape: {df_clean.shape}")
assert df_clean.shape[0] == 56373, f"Unexpected final row count: {df_clean.shape[0]}"

# 1. Ensure primary key / UniqueID uniqueness
assert df_clean['UniqueID'].is_unique, "UniqueID is not unique"

# 2. Ensure no business-key duplicates remain
key_cols = ["ParcelID", "PropertyAddress", "SaleDate", "SalePrice", "LegalReference"]
assert df_clean.duplicated(subset=key_cols).sum() == 0, "Business-key duplicates still exist"

# 3. Ensure PropertyAddress has zero nulls
assert df_clean['PropertyAddress'].isna().sum() == 0, "PropertyAddress contains nulls"

# 4. Ensure SoldAsVacant only has Yes/No
assert set(df_clean['SoldAsVacant'].unique()) == {'Yes', 'No'}, "Invalid values in SoldAsVacant"
assert df_clean['SoldAsVacant'].isna().sum() == 0, "SoldAsVacant contains nulls"

# 5. Ensure flag columns are boolean and non-null
flag_cols = [
    'Flag_LowSalePrice', 'Flag_ZeroBedrooms', 
    'Flag_LargeAcreage', 'Flag_UnusualYearBuilt', 'HasAssessorData'
]
for col in flag_cols:
    assert col in df_clean.columns, f"Missing flag column {col}"
    assert df_clean[col].dtype == bool, f"{col} is not boolean"
    assert df_clean[col].isna().sum() == 0, f"{col} contains nulls"

# 6. Ensure change_log is populated correctly
assert len(change_log) > 0, "change_log is empty"
display(change_log)

print("All final validation checks passed successfully.")

Final shape: (56373, 29)


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


All final validation checks passed successfully.


## Step 9 - Export Cleaned Data and Change Log

Save the cleaned dataset to Excel and CSV formats, save the change log, and validate the exported files.

In [37]:
# Step 9a: Export data and logs
import os

os.makedirs("../data/cleaned", exist_ok=True)
os.makedirs("../reports", exist_ok=True)

csv_path = "../data/cleaned/nashville_housing_cleaned.csv"
excel_path = "../data/cleaned/nashville_housing_cleaned.xlsx"
changelog_path = "../reports/change_log.csv"

df_clean.to_csv(csv_path, index=False, encoding="utf-8")
df_clean.to_excel(excel_path, index=False)
change_log.to_csv(changelog_path, index=False, encoding="utf-8")

print("Files exported successfully.")

Files exported successfully.


In [38]:
# Step 9b: Validate exported files
df_reloaded_csv = pd.read_csv(csv_path)
df_reloaded_excel = pd.read_excel(excel_path)
changelog_reloaded = pd.read_csv(changelog_path)

assert df_reloaded_csv.shape == df_clean.shape, f"CSV shape mismatch: {df_reloaded_csv.shape} vs {df_clean.shape}"
assert df_reloaded_excel.shape == df_clean.shape, f"Excel shape mismatch: {df_reloaded_excel.shape} vs {df_clean.shape}"
assert list(df_reloaded_csv.columns) == list(df_clean.columns), "CSV columns mismatch"
assert list(df_reloaded_excel.columns) == list(df_clean.columns), "Excel columns mismatch"
assert len(changelog_reloaded) == len(change_log), "Change log rows mismatch"

print("All export validations passed successfully. Data cleaning pipeline is fully complete.")
display(change_log)

All export validations passed successfully. Data cleaning pipeline is fully complete.


,step,description,rows_affected
0,Step 1a,Strip and collapse whitespace in PropertyAddress,55863
1,Step 1a,Strip and collapse whitespace in OwnerAddress,25315
2,Step 1b,LandUse: strip and collapse whitespace,38
3,Step 1b,LandUse: 'VACANT RES LAND' -> 'VACANT RESIDENT...,1549
4,Step 1b,LandUse: 'VACANT RESIENTIAL LAND' -> 'VACANT R...,3
5,Step 1b,LandUse: 'RESTURANT/CAFETERIA' -> 'RESTAURANT/...,2
6,Step 1b,LandUse: 'GRRENBELT...' -> 'GREENBELT/RES',3
7,Step 1c,Rename column header 'UniqueID ' (trailing spa...,1
8,Step 2,"SoldAsVacant: 'Y' -> 'Yes', 'N' -> 'No'",451
9,Step 3,Fill missing PropertyAddress via ParcelID (29 ...,29


## Appendix: Issue Register

In [39]:
# Build the issue register from live data (every number comes from df_raw / df_clean)
from pathlib import Path

REPORTS = Path("../reports")
n_raw, n_clean = len(df_raw), len(df_clean)
rows = []

def add(issue_id, columns, issue_type, evidence, count, severity, action_taken,
        client_decision=False, base=None):
    """Append one issue; pct_rows uses the df_raw size unless another base is given."""
    base = base or n_raw
    rows.append({
        "id": issue_id, "columns": columns, "issue_type": issue_type,
        "evidence": evidence, "rows_affected": int(count),
        "pct_rows": round(100 * int(count) / base, 2), "severity": severity,
        "action_taken": action_taken, "needs_client_decision": client_decision,
    })

def ws_dirty(s):
    """Rows whose text differs from its trimmed, single-spaced version (NaN ignored)."""
    cleaned = s.str.strip().str.replace(r"\s+", " ", regex=True)
    return int(((s != cleaned) & s.notna()).sum())

raw_land = df_raw["LandUse"]
land_variants = (raw_land.isin(["VACANT RES LAND", "VACANT RESIENTIAL LAND", "RESTURANT/CAFETERIA"])
                 | raw_land.str.contains("GRRENBELT", na=False))
conflicts = pd.read_csv(REPORTS / "parcel_address_conflicts.csv")
uid = df_raw["UniqueID "]   # the raw header has a trailing space

add("I01", "SoldAsVacant", "Inconsistent labels", "'Y'/'N' used next to 'Yes'/'No'",
    df_raw["SoldAsVacant"].isin(["Y", "N"]).sum(), "Medium", "Standardized to Yes/No")
add("I02", "PropertyAddress", "Extra spaces", "Repeated or leading/trailing spaces",
    ws_dirty(df_raw["PropertyAddress"]), "Low", "Trimmed and collapsed spaces")
add("I03", "OwnerAddress", "Extra spaces", "Repeated spaces",
    ws_dirty(df_raw["OwnerAddress"]), "Low", "Trimmed and collapsed spaces")
add("I04", "LandUse", "Misspelled or split labels",
    "'VACANT RES LAND', 'VACANT RESIENTIAL LAND', 'RESTURANT/CAFETERIA', 'GRRENBELT' variant",
    land_variants.sum(), "Medium", "Mapped to one standard label each")
add("I05", "PropertyAddress", "Missing values", "Empty address",
    df_raw["PropertyAddress"].isna().sum(), "Medium", "Filled from the same ParcelID")
add("I06", "OwnerAddress, Acreage, LandValue, BuildingValue, TotalValue, TaxDistrict",
    "Structural missing values", "All six columns empty in exactly the same rows",
    df_raw["TotalValue"].isna().sum(), "High",
    "Not imputed; added HasAssessorData flag", True)
add("I07", "ParcelID, PropertyAddress, SaleDate, SalePrice, LegalReference",
    "Duplicate sales", "Same parcel, address, date, price and legal reference",
    n_raw - n_clean, "High", "Removed, keeping the lowest UniqueID", True)
add("I08", "ParcelID, PropertyAddress", "Parcel with several addresses",
    f"{conflicts['ParcelID'].nunique()} ParcelIDs appear with more than one address",
    len(conflicts), "Medium", "Not changed; evidence in parcel_address_conflicts.csv", True)
add("I09", "LandUse", "Possibly overlapping categories",
    "'CONDO' and 'RESIDENTIAL CONDO' may be the same category",
    (df_clean["LandUse"] == "CONDO").sum(), "Low", "Left separate", True, n_clean)
add("I10", "LandUse, SoldAsVacant", "Possible mismatch",
    "Vacant-land LandUse but SoldAsVacant = No",
    (df_clean["LandUse"].str.contains("VACANT", na=False)
     & (df_clean["SoldAsVacant"] == "No")).sum(),
    "Low", "Not changed; reported", True, n_clean)

for fid, col, desc in [
    ("I11", "Flag_LowSalePrice", "SalePrice below 10,000"),
    ("I12", "Flag_ZeroBedrooms", "Zero bedrooms on a residential land use"),
    ("I13", "Flag_LargeAcreage", "Acreage above 10"),
    ("I14", "Flag_UnusualYearBuilt", "Built more than 1 year after the sale year"),
]:
    add(fid, col, "Suspicious value", desc, df_clean[col].sum(), "Low",
        "Flagged only, nothing removed", False, n_clean)

add("I15", "Bedrooms, YearBuilt, FullBath, HalfBath", "Many missing values",
    "Property details not recorded", df_raw["Bedrooms"].isna().sum(), "Medium",
    "Not imputed; reported", True)
add("I16", "UniqueID", "Gaps in ID sequence", "IDs missing between min and max (count of IDs)",
    uid.max() - uid.min() + 1 - uid.nunique(), "Low",
    "Not changed; noted for the client", False)
add("I17", "UniqueID", "Column header has a trailing space",
    "Raw header is 'UniqueID ' (counts 1 column, not rows)", 1, "Low",
    "Renamed to UniqueID", False)

issues = pd.DataFrame(rows)
issues.to_csv(REPORTS / "issue_register.csv", index=False)
display(issues[["id", "issue_type", "columns", "rows_affected", "pct_rows", "severity"]])

,id,issue_type,columns,rows_affected,pct_rows,severity
0,I01,Inconsistent labels,SoldAsVacant,451,0.80,Medium
1,I02,Extra spaces,PropertyAddress,55863,98.91,Low
2,I03,Extra spaces,OwnerAddress,25315,44.82,Low
3,I04,Misspelled or split labels,LandUse,1557,2.76,Medium
4,I05,Missing values,PropertyAddress,29,0.05,Medium
5,I06,Structural missing values,"OwnerAddress, Acreage, LandValue, BuildingValu...",30462,53.94,High
6,I07,Duplicate sales,"ParcelID, PropertyAddress, SaleDate, SalePrice...",104,0.18,High
7,I08,Parcel with several addresses,"ParcelID, PropertyAddress",526,0.93,Medium
8,I09,Possibly overlapping categories,LandUse,247,0.44,Low
9,I10,Possible mismatch,"LandUse, SoldAsVacant",1296,2.30,Low


In [40]:
# Evidence for the open questions in the report (READ-ONLY)
print("=== I09: CONDO vs RESIDENTIAL CONDO ===")
condo = df_clean[df_clean["LandUse"].isin(["CONDO", "RESIDENTIAL CONDO"])]
display(condo.groupby("LandUse").agg(
    rows=("UniqueID", "size"),
    share_with_assessor_data=("HasAssessorData", "mean"),
    share_sold_vacant=("SoldAsVacant", lambda s: (s == "Yes").mean()),
    median_price=("SalePrice", "median"),
    median_year_built=("YearBuilt", "median"),
    first_sale=("SaleDate", "min"),
    last_sale=("SaleDate", "max"),
))
both = (set(df_clean.loc[df_clean["LandUse"] == "CONDO", "ParcelID"])
        & set(df_clean.loc[df_clean["LandUse"] == "RESIDENTIAL CONDO", "ParcelID"]))
print("ParcelIDs that appear under BOTH labels:", len(both))

print("\n=== I10: LandUse vs SoldAsVacant ===")
is_vacant_use = df_clean["LandUse"].str.contains("VACANT", na=False).rename("LandUse says vacant")
display(pd.crosstab(is_vacant_use, df_clean["SoldAsVacant"], margins=True))
print("Sold as vacant (Yes) but NOT a vacant land use, top 5:")
print(df_clean.loc[(df_clean["SoldAsVacant"] == "Yes") & ~is_vacant_use, "LandUse"]
      .value_counts().head(5).to_string())

print("\n=== I16: gaps in UniqueID (raw file) ===")
uid = df_raw["UniqueID "]
missing_ids = sorted(set(range(int(uid.min()), int(uid.max()) + 1)) - set(uid))
print("Missing IDs:", len(missing_ids), "| first 10:", missing_ids[:10])

print("\n=== I14: numbers behind the build-year note ===")
flag = df_clean["Flag_UnusualYearBuilt"]
has_year = df_clean["YearBuilt"].notna()
print("Yes rate, all rows:", round((df_clean["SoldAsVacant"] == "Yes").mean(), 3))
print("Yes rate, rows with a build year:",
      round((df_clean.loc[has_year, "SoldAsVacant"] == "Yes").mean(), 3))
gap = (df_clean["YearBuilt"] - df_clean["SaleDate"].dt.year)[flag]
print("Year gap (build year minus sale year), flagged rows:")
print(gap.value_counts().sort_index().to_string())
print("LandUse of flagged rows with SoldAsVacant == 'No':")
print(df_clean.loc[flag & (df_clean["SoldAsVacant"] == "No"), "LandUse"]
      .value_counts().head(5).to_string())

=== I09: CONDO vs RESIDENTIAL CONDO ===


,rows,share_with_assessor_data,share_sold_vacant,median_price,median_year_built,first_sale,last_sale
LandUse,,,,,,,
CONDO,247,0.0,0.000000,362000.0,NaN,2013-01-16,2014-05-30
RESIDENTIAL CONDO,14064,0.0,0.015856,220000.0,NaN,2013-01-02,2016-10-31


ParcelIDs that appear under BOTH labels: 54

=== I10: LandUse vs SoldAsVacant ===


SoldAsVacant,No,Yes,All
LandUse says vacant,,,
False,50408,852,51260
True,1296,3817,5113
All,51704,4669,56373


Sold as vacant (Yes) but NOT a vacant land use, top 5:
LandUse
SINGLE FAMILY             593
RESIDENTIAL CONDO         223
RESIDENTIAL COMBO/MISC     13
DUPLEX                      9
FOREST                      3

=== I16: gaps in UniqueID (raw file) ===
Missing IDs: 159 | first 10: [1124, 1713, 1767, 1820, 1842, 1852, 1887, 1891, 1911, 2004]

=== I14: numbers behind the build-year note ===
Yes rate, all rows: 0.083
Yes rate, rows with a build year: 0.018
Year gap (build year minus sale year), flagged rows:
2.0    147
3.0     27
4.0      1
LandUse of flagged rows with SoldAsVacant == 'No':
LandUse
SINGLE FAMILY              87
VACANT RESIDENTIAL LAND    29
DUPLEX                      4
TRIPLEX                     2
RESIDENTIAL COMBO/MISC      2
